# Air Quality Analysis and Prediction (ES 678): all code in one notebook

Next-day PM2.5 forecasting for Indian cities (CPCB data, 2015-2020). This notebook contains **every function and every experiment** of the project, in the order they run. Run all cells top to bottom (about 10 minutes on Colab's CPU).

**How to use it in Google Colab**
1. Open the notebook in Colab (Runtime > Run all).
2. When the data-loading cell asks, upload `city_day.csv` (download *Air Quality Data in India (2015-2020)* from Kaggle first; a free account is needed).
3. Outputs are written to `results/` and `paper/` inside the Colab session; download them from the Files panel if you want them.

**Playing around:** all settings are in the *Configuration* cell (split dates, city threshold, grids, interval level, bootstrap size). Change one, then re-run from that cell downwards. A small plotting cell at the very end lets you inspect any city.

# Air Quality Analysis and Prediction: next-day PM2.5 forecasting (ES 678)

This file holds every function used in the project. `scripts/run_all.py` runs the
experiments, and `notebooks/air_quality_colab.ipynb` is built from these same two files.

**Question.** Can today's pollutant readings and calendar information predict tomorrow's
city-level PM2.5 in Indian cities more accurately than persistence (tomorrow = today)?
One row is one city on one day. Everything is built so that the features of a row use only
information available at the end of that day, and every learned step is fit on training data only.

In [ ]:
import json
import os
import platform
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:  # running inside a notebook
    ROOT = Path(".")

## 0. Configuration
All choices that matter are set here (and nowhere else), so they can be audited or changed.

In [ ]:
SEED = 42

# Chronological split, defined by the date of the TARGET (the day being predicted).
TRAIN_END = "2018-12-31"      # train: 2015-2018
VAL_END = "2019-12-31"        # validation (tuning + conformal calibration): 2019
TEST_END = "2020-07-01"       # test: 2020 (the file ends on 2020-07-01); touched once
LOCKDOWN_START = "2020-03-25" # India's nationwide COVID-19 lockdown began on this date

MIN_TRAIN_ROWS = 200      # a city needs this many usable training rows to be kept
MAX_MISSING_TRAIN = 0.50  # drop a numeric feature if more than this share is missing in TRAIN
FFILL_LIMIT = 3           # past-only forward fill, at most 3 days
HIGH_Q = 0.90             # "high-pollution" = target above this quantile of TRAINING targets
ALPHA = 0.10              # intervals have 90% nominal coverage
BLOCK = 7                 # block length (days) for the moving-block bootstrap
NBOOT = 2000

POLL = ["PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene"]
CAL = ["month_sin", "month_cos", "dow_sin", "dow_cos"]
BASE = ["pm25_today"] + POLL + CAL + ["City"]
FEATURE_SETS = {
    "today only": BASE,
    "lags 1-3": BASE + ["pm25_lag1", "pm25_lag2", "pm25_lag3", "pm25_roll3"],
    "lags 1-7": BASE + [f"pm25_lag{k}" for k in range(1, 8)] + ["pm25_roll3", "pm25_roll7"],
}
MAIN_SET = "lags 1-7"
MODEL_NAMES = ["ridge", "rf", "gb"]
MODEL_LABELS = {"persistence": "Persistence", "city-month": "City-month mean",
                "ridge": "Ridge", "rf": "Random forest", "gb": "Gradient boosting"}

# Hyperparameter grids, tuned on the 2019 validation year only.
GRIDS = {
    "ridge": [{"alpha": a} for a in (0.1, 1, 10, 100, 1000)],
    "rf": [{"max_depth": d, "min_samples_leaf": l} for d in (8, 14, None) for l in (3, 10)],
    "gb": [{"n_estimators": n, "learning_rate": lr, "max_depth": md}
           for n in (150, 400) for lr in (0.03, 0.1) for md in (2, 4)],
}

## 1. Data loading and quality audit

In [ ]:
def find_data(path=None):
    """Locate city_day.csv. In Google Colab, offer an upload dialog if it is not found."""
    cands = [path, os.environ.get("AQ_DATA"), ROOT / "data" / "city_day.csv", "city_day.csv",
             "data/city_day.csv", "/content/city_day.csv", "../data/city_day.csv"]
    for c in cands:
        if c and Path(c).exists():
            return str(c)
    try:
        from google.colab import files  # type: ignore
        print("city_day.csv not found. Please upload it.")
        up = files.upload()
        return [k for k in up if k.endswith(".csv")][0]
    except ImportError:
        raise FileNotFoundError(
            "city_day.csv not found. Download 'Air Quality Data in India (2015-2020)' from Kaggle "
            "and place city_day.csv in data/ (see data/README.md).")


def load_raw(path):
    raw = pd.read_csv(path, parse_dates=["Date"])
    return raw.sort_values(["City", "Date"]).reset_index(drop=True)


def to_daily(raw):
    """Reindex every city to a complete daily calendar so that lags are in calendar days."""
    cols = ["PM2.5"] + POLL
    parts = []
    for city, g in raw.groupby("City"):
        g = g.set_index("Date")[cols].sort_index()
        idx = pd.date_range(g.index.min(), g.index.max(), freq="D")
        g = g.reindex(idx)
        g.insert(0, "City", city)
        g.index.name = "Date"
        parts.append(g.reset_index())
    return pd.concat(parts, ignore_index=True)


def audit(raw, daily):
    """Data-quality checks reported in the paper: missingness, duplicates, gaps, extremes."""
    obs = raw.dropna(subset=["PM2.5"])
    miss = (raw[["PM2.5"] + POLL + ["AQI"]].isna().mean() * 100).round(1).rename("missing_pct")
    per_city = pd.DataFrame({
        "rows": raw.groupby("City").size(),
        "calendar_days": daily.groupby("City").size(),
        "pm25_observed": obs.groupby("City").size(),
    }).fillna(0).astype(int)
    per_city["missing_calendar_days"] = per_city["calendar_days"] - per_city["rows"]
    by_year = obs.pivot_table(index="City", columns=obs["Date"].dt.year, values="PM2.5",
                              aggfunc="count").fillna(0).astype(int)
    pm = obs["PM2.5"]
    summary = {
        "rows": int(len(raw)), "cities": int(raw["City"].nunique()),
        "date_min": str(raw["Date"].min().date()), "date_max": str(raw["Date"].max().date()),
        "duplicate_city_dates": int(raw.duplicated(["City", "Date"]).sum()),
        "pm25_missing_pct": float(round(raw["PM2.5"].isna().mean() * 100, 1)),
        "pm25_min": float(pm.min()), "pm25_median": float(pm.median()),
        "pm25_p99": float(pm.quantile(0.99)), "pm25_max": float(pm.max()),
        "pm25_nonpositive": int((pm <= 0).sum()),
        "calendar_days_missing_total": int(per_city["missing_calendar_days"].sum()),
    }
    return summary, miss, per_city, by_year

## 2. Leak-free features and target
* Features of the row dated *t* use only data up to day *t* (past-only forward fill,
  lags >= 0, trailing rolling means).
* The target is PM2.5 on day *t+1*. Rows need an observed PM2.5 on day *t* and on day *t+1*, so that
  persistence is defined and every method is scored on exactly the same rows.
* `AQI` and `AQI_Bucket` are never used (AQI is computed from the same pollutants).

In [ ]:
def build_features(daily):
    d = daily.sort_values(["City", "Date"]).reset_index(drop=True)
    cols = ["PM2.5"] + POLL
    ff = d.groupby("City")[cols].ffill(limit=FFILL_LIMIT)  # past-only by construction
    city = d["City"]
    X = pd.DataFrame({"City": city, "Date": d["Date"]})
    X["pm25_today"] = ff["PM2.5"]
    for p in POLL:
        X[p] = ff[p]
    pm = ff["PM2.5"]
    for k in range(1, 8):
        X[f"pm25_lag{k}"] = pm.groupby(city).shift(k)
    for w in (3, 7):
        X[f"pm25_roll{w}"] = pm.groupby(city).transform(lambda s, w=w: s.rolling(w, min_periods=1).mean())
    m, wd = d["Date"].dt.month, d["Date"].dt.dayofweek
    X["month_sin"], X["month_cos"] = np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)
    X["dow_sin"], X["dow_cos"] = np.sin(2 * np.pi * wd / 7), np.cos(2 * np.pi * wd / 7)
    X["y_today"] = d["PM2.5"]                              # raw observed PM2.5 today
    X["y"] = d.groupby("City")["PM2.5"].shift(-1)          # raw observed PM2.5 tomorrow (target)
    X["target_date"] = d["Date"] + pd.Timedelta(days=1)
    return X


def make_dataset(daily):
    F = build_features(daily)
    return F.dropna(subset=["y_today", "y"]).reset_index(drop=True)


def split(F, train_end=TRAIN_END, val_end=VAL_END, test_end=TEST_END):
    """Chronological split by TARGET date, so no target date falls in two splits."""
    td = F["target_date"]
    tr = F[td <= pd.Timestamp(train_end)]
    va = F[(td > pd.Timestamp(train_end)) & (td <= pd.Timestamp(val_end))]
    te = F[(td > pd.Timestamp(val_end)) & (td <= pd.Timestamp(test_end))]
    return tr.copy(), va.copy(), te.copy()


def select_numeric(train, cols, thr=MAX_MISSING_TRAIN):
    """Drop numeric features that are mostly missing IN TRAINING. Learned from train only."""
    num = [c for c in cols if c != "City"]
    miss = train[num].isna().mean()
    kept = [c for c in num if miss[c] <= thr]
    return kept, miss


def cols_for(train, set_name, kept_num=None):
    if kept_num is None:
        kept_num, _ = select_numeric(train, FEATURE_SETS[MAIN_SET])
    return [c for c in FEATURE_SETS[set_name] if c == "City" or c in kept_num]

## 3. Baselines and models
* **Persistence**: tomorrow = today.
* **City-month mean**: mean target for the same city and calendar month, computed on training data only.
* **Ridge, random forest, gradient boosting**, inside a pipeline whose imputer (train medians),
  scaler (ridge only) and one-hot encoder are fit on training data only.

In [ ]:
def persistence(df):
    return df["y_today"].to_numpy(dtype=float)


def city_month_fit(train):
    t = train.assign(m=train["target_date"].dt.month)
    return t.groupby(["City", "m"])["y"].mean(), t.groupby("m")["y"].mean(), float(train["y"].mean())


def city_month_predict(model, df):
    cm, mm, g = model
    m = df["target_date"].dt.month
    idx = pd.MultiIndex.from_arrays([df["City"], m])
    p = pd.Series(cm.reindex(idx).to_numpy(), index=df.index)
    return p.fillna(m.map(mm)).fillna(g).to_numpy(dtype=float)


def _ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # older scikit-learn
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_pipe(cols, model, scale):
    num = [c for c in cols if c != "City"]
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    pre = ColumnTransformer([("num", Pipeline(steps), num), ("city", _ohe(), ["City"])])
    return Pipeline([("pre", pre), ("model", model)])


def make_model(name, hp, seed=SEED):
    if name == "ridge":
        return Ridge(**hp)
    if name == "rf":
        return RandomForestRegressor(n_estimators=300, max_features=0.5, n_jobs=-1, random_state=seed, **hp)
    if name == "gb":
        return GradientBoostingRegressor(subsample=0.8, random_state=seed, **hp)
    raise ValueError(name)


def fit_model(name, hp, cols, train):
    pipe = build_pipe(cols, make_model(name, hp), scale=(name == "ridge"))
    return pipe.fit(train[cols], train["y"])


def tune(name, cols, train, val):
    """Grid search on the validation year only. Returns best hyperparameters and the full table."""
    rows = []
    for hp in GRIDS[name]:
        m = fit_model(name, hp, cols, train)
        rows.append({**hp, "val_mae": mae(val["y"].to_numpy(), m.predict(val[cols]))})
    tab = pd.DataFrame(rows)
    best = GRIDS[name][int(tab["val_mae"].idxmin())]
    return best, tab

## 4. Metrics and uncertainty of the comparison
MAE and RMSE are in PM2.5 units (ug/m3). "High-pollution days" are days whose target is above the
90th percentile of the TRAINING targets. The skill score is `1 - MAE_model / MAE_persistence`
(positive = better than persistence). Confidence intervals come from a moving-block bootstrap over
test dates (blocks of 7 days, all cities of a date resampled together), because consecutive days
are strongly correlated.

In [ ]:
def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(p, float))))


def _safe(a, b):
    return a / b if b > 0 else np.nan


def metrics(df, pred, hi_thr, ref=None):
    y = df["y"].to_numpy(float)
    e = np.asarray(pred, float) - y
    hi = y >= hi_thr
    out = {"n": int(len(y)), "MAE": float(np.mean(np.abs(e))), "RMSE": float(np.sqrt(np.mean(e ** 2))),
           "n_high": int(hi.sum()), "MAE_high": float(np.mean(np.abs(e[hi]))) if hi.any() else np.nan,
           "bias": float(np.mean(e))}
    if ref is not None:
        out["skill"] = 1 - out["MAE"] / float(np.mean(np.abs(np.asarray(ref, float) - y)))
    return out


def block_bootstrap(dates, arrays, stat_fns, B=NBOOT, block=BLOCK, seed=SEED, level=0.95):
    """Moving-block bootstrap over dates. `arrays` are per-row values, summed per date."""
    u, inv = np.unique(np.asarray(dates), return_inverse=True)
    D = len(u)
    block = min(block, D)
    agg = {k: np.bincount(inv, weights=np.asarray(v, float), minlength=D) for k, v in arrays.items()}
    agg["n"] = np.bincount(inv, minlength=D).astype(float)
    tot = {k: v.sum() for k, v in agg.items()}
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(D / block))
    draws = {k: np.empty(B) for k in stat_fns}
    for b in range(B):
        starts = rng.integers(0, D - block + 1, size=nb)
        idx = (starts[:, None] + np.arange(block)).ravel()[:D]
        s = {k: v[idx].sum() for k, v in agg.items()}
        for name, fn in stat_fns.items():
            draws[name][b] = fn(s)
    out = {}
    for name, fn in stat_fns.items():
        lo, hi = np.nanquantile(draws[name], [(1 - level) / 2, 1 - (1 - level) / 2])
        out[name] = (float(fn(tot)), float(lo), float(hi))
    return out


COMPARE_STATS = {
    "MAE": lambda s: _safe(s["am"], s["n"]),
    "skill": lambda s: 1 - _safe(s["am"], s["ap"]),
    "dMAE": lambda s: _safe(s["am"] - s["ap"], s["n"]),
    "skill_high": lambda s: 1 - _safe(s["amh"], s["aph"]),
}


def boot_vs_persistence(df, pred, hi_thr):
    y = df["y"].to_numpy(float)
    am, ap = np.abs(np.asarray(pred, float) - y), np.abs(persistence(df) - y)
    hi = (y >= hi_thr).astype(float)
    return block_bootstrap(df["target_date"].to_numpy(),
                           {"am": am, "ap": ap, "amh": am * hi, "aph": ap * hi}, COMPARE_STATS)


def evaluate(df, preds, hi_thr, with_ci=True):
    """One row per model. `preds` must contain 'persistence' (the reference)."""
    rows = []
    for name, p in preds.items():
        r = {"model": name, **metrics(df, p, hi_thr, ref=preds["persistence"])}
        if with_ci:
            ci = boot_vs_persistence(df, p, hi_thr)
            for k in ("MAE", "skill", "dMAE", "skill_high"):
                r[k + "_lo"], r[k + "_hi"] = ci[k][1], ci[k][2]
        rows.append(r)
    return pd.DataFrame(rows)

## 5. Error and failure-mode analysis helpers

In [ ]:
def group_errors(df, pred, groups):
    """Bias, MAE and under-prediction share for named boolean masks (a dict name -> mask)."""
    y = df["y"].to_numpy(float)
    p = np.asarray(pred, float)
    e = p - y
    rows = []
    for name, mask in groups.items():
        mask = np.asarray(mask, bool)
        if mask.sum() == 0:
            continue
        rows.append({"group": name, "n": int(mask.sum()), "MAE": float(np.mean(np.abs(e[mask]))),
                     "bias": float(np.mean(e[mask])),
                     "share_underpredicted": float(np.mean(e[mask] < 0)),
                     "pred_over_obs": float(p[mask].mean() / y[mask].mean())})
    return pd.DataFrame(rows)


def per_city_table(df, pred, ref):
    t = df[["City", "y"]].copy()
    t["am"], t["ap"] = np.abs(np.asarray(pred, float) - t["y"]), np.abs(np.asarray(ref, float) - t["y"])
    g = t.groupby("City").agg(n=("y", "size"), MAE_model=("am", "mean"), MAE_persistence=("ap", "mean"))
    g["skill"] = 1 - g["MAE_model"] / g["MAE_persistence"]
    return g.reset_index()

## 6. Prediction intervals: quantile gradient boosting and split-conformal
* **Quantile gradient boosting** fits the 5% and 95% conditional quantiles directly (no calibration).
* **Split-conformal** uses absolute residuals of the point model on a calibration year and returns
  `prediction +/- q` where `q` is the `ceil((n+1)(1-alpha))/n` empirical quantile.
  Its coverage guarantee assumes exchangeability, which a time series does not satisfy, so coverage is
  measured on later periods (rolling windows and the 2020 lockdown) instead of being assumed.

In [ ]:
def conformal_q(residuals, alpha=ALPHA):
    r = np.sort(np.asarray(residuals, float))
    n = len(r)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(r[min(k, n) - 1])


def fit_quantile_gb(hp, cols, train, q):
    model = GradientBoostingRegressor(loss="quantile", alpha=q, subsample=0.8, random_state=SEED, **hp)
    return build_pipe(cols, model, scale=False).fit(train[cols], train["y"])


WINDOWS = [  # (label, train_end, calibration_end, test_end): train < calibration < test, all by target date
    ("2018 backtest", "2016-12-31", "2017-12-31", "2018-12-31"),
    ("2019 backtest", "2017-12-31", "2018-12-31", "2019-12-31"),
    ("2020 test", "2018-12-31", "2019-12-31", "2020-07-01"),
]


def coverage_rows(window, method, df, lo, hi, hi_thr, extra_groups=None):
    y = df["y"].to_numpy(float)
    covered = ((y >= lo) & (y <= hi)).astype(float)
    width = np.clip(hi, 0, None) - np.clip(lo, 0, None)
    high = y >= hi_thr
    groups = {"all": np.ones(len(y), bool), "high-pollution days": high}
    groups.update(extra_groups or {})
    rows = []
    for gname, mask in groups.items():
        if mask.sum() == 0:
            continue
        r = {"window": window, "method": method, "group": gname, "n": int(mask.sum()),
             "coverage": float(covered[mask].mean()), "mean_width": float(width[mask].mean())}
        if gname == "all":
            ci = block_bootstrap(df["target_date"].to_numpy(), {"c": covered},
                                 {"cov": lambda s: _safe(s["c"], s["n"])}, B=1000)["cov"]
            r["coverage_lo"], r["coverage_hi"] = ci[1], ci[2]
        rows.append(r)
    return rows

## 7. Plain-text helpers for saving results and LaTeX tables

In [ ]:
def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))


def versions():
    return {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "seed": SEED}


def fmt(x, nd=1):
    return "--" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{nd}f}"

# Run the full experiment
Executes the pipeline end to end: data audit -> chronological split -> leak-free features ->
baselines and models -> tuning on 2019 -> single evaluation on 2020 -> graduate-depth analyses
(lag ablation, failure audit, prediction intervals) -> figures and LaTeX tables.

In [ ]:
import time

T0 = time.time()
RESULTS = ROOT / "results"
TABLES = ROOT / "paper" / "tables"
FIGS = ROOT / "paper" / "figures"
for p in (RESULTS, TABLES, FIGS):
    p.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
SUMMARY = {"versions": versions()}

## 1. Load data and audit quality

In [ ]:
DATA = find_data()
raw = load_raw(DATA)
daily = to_daily(raw)
a_sum, a_miss, a_city, a_year = audit(raw, daily)
SUMMARY["data_audit"] = a_sum
print(json.dumps(a_sum, indent=2))
print("\nMissing % per column:\n", a_miss.to_string())
a_miss.to_csv(RESULTS / "audit_missing_pct.csv")
a_city.to_csv(RESULTS / "audit_per_city.csv")
a_year.to_csv(RESULTS / "audit_pm25_counts_city_year.csv")

## 2. Dataset, city filter and chronological split
Cities with fewer than `MIN_TRAIN_ROWS` usable training rows are dropped (this uses training data only).

In [ ]:
F_all = make_dataset(daily)
tr0, _, _ = split(F_all)
counts = tr0["City"].value_counts()
KEEP = sorted(counts[counts >= MIN_TRAIN_ROWS].index)
DROPPED = sorted(set(F_all["City"].unique()) - set(KEEP))
F = F_all[F_all["City"].isin(KEEP)].reset_index(drop=True)
train, val, test = split(F)

# Hard checks that the split is chronological and non-overlapping.
assert train["target_date"].max() <= pd.Timestamp(TRAIN_END)
assert val["target_date"].min() > train["target_date"].max()
assert test["target_date"].min() > val["target_date"].max()

HI = float(np.quantile(train["y"], HIGH_Q))
LOCK = pd.Timestamp(LOCKDOWN_START)
pre_mask = (test["target_date"] < LOCK).to_numpy()
lock_mask = ~pre_mask

city_rows = pd.DataFrame({
    "train_rows": train.groupby("City").size(), "val_rows": val.groupby("City").size(),
    "test_rows": test.groupby("City").size()}).fillna(0).astype(int)
city_rows.to_csv(RESULTS / "kept_cities_rows.csv")
SUMMARY["split"] = {
    "rows_before_filter": int(len(F_all)), "cities_kept": KEEP, "cities_dropped": DROPPED,
    "train_rows": len(train), "val_rows": len(val), "test_rows": len(test),
    "test_pre_lockdown_rows": int(pre_mask.sum()), "test_lockdown_rows": int(lock_mask.sum()),
    "high_threshold_ugm3": HI, "high_days_test": int((test["y"] >= HI).sum()),
    "train_target_dates": [str(train["target_date"].min().date()), str(train["target_date"].max().date())],
    "val_target_dates": [str(val["target_date"].min().date()), str(val["target_date"].max().date())],
    "test_target_dates": [str(test["target_date"].min().date()), str(test["target_date"].max().date())],
}
print(f"kept {len(KEEP)} cities, dropped {len(DROPPED)}: {DROPPED}")
print(f"rows: train {len(train)}, val {len(val)}, test {len(test)} (pre-lockdown {pre_mask.sum()}, lockdown {lock_mask.sum()})")
print(f"high-pollution threshold (train 90th percentile): {HI:.1f} ug/m3; high days in test: {(test['y'] >= HI).sum()}")
print(city_rows.to_string())

## 3. Baselines, tuning on validation (2019), final evaluation on test (2020)

In [ ]:
KEPT_NUM, miss_train = select_numeric(train, FEATURE_SETS[MAIN_SET])
DROPPED_FEATURES = [c for c in miss_train.index if c not in KEPT_NUM]
COLS = cols_for(train, MAIN_SET, KEPT_NUM)
SUMMARY["features"] = {"kept_numeric": KEPT_NUM, "dropped_for_missingness_in_train": DROPPED_FEATURES,
                       "train_missing_share": miss_train.round(3).to_dict()}
print("dropped features (missing share in train > %.0f%%):" % (MAX_MISSING_TRAIN * 100), DROPPED_FEATURES)

cm = city_month_fit(train)
BEST_HP, TUNING = {}, {}
for name in MODEL_NAMES:
    BEST_HP[name], TUNING[name] = tune(name, COLS, train, val)
    TUNING[name].to_csv(RESULTS / f"tuning_{name}.csv", index=False)
    print(f"{name}: best {BEST_HP[name]}  val MAE {TUNING[name]['val_mae'].min():.2f}")
FITTED = {n: fit_model(n, BEST_HP[n], COLS, train) for n in MODEL_NAMES}


def predict_all(df):
    p = {"persistence": persistence(df), "city-month": city_month_predict(cm, df)}
    for n in MODEL_NAMES:
        p[n] = FITTED[n].predict(df[COLS])
    return p


P_val, P_test = predict_all(val), predict_all(test)
pred_out = test[["City", "target_date", "y_today", "y"]].reset_index(drop=True).copy()
for k, v in P_test.items():
    pred_out[f"pred_{k}"] = v
pred_out.to_csv(RESULTS / "test_predictions.csv", index=False)
val_tab = evaluate(val, P_val, HI, with_ci=False)
BEST = val_tab[val_tab["model"].isin(MODEL_NAMES)].sort_values("MAE").iloc[0]["model"]
print("\nVALIDATION 2019\n", val_tab.round(3).to_string(index=False))
print("\nbest learned model on validation:", BEST)

# The test set is evaluated here, once, after models and hyperparameters are frozen.
test_tab = evaluate(test, P_test, HI, with_ci=True)
pre_tab = evaluate(test[pre_mask], {k: v[pre_mask] for k, v in P_test.items()}, HI, with_ci=True)
lock_tab = evaluate(test[lock_mask], {k: v[lock_mask] for k, v in P_test.items()}, HI, with_ci=True)
test_tab.assign(period="all 2020 test").to_csv(RESULTS / "test_metrics_all.csv", index=False)
pre_tab.assign(period="pre-lockdown").to_csv(RESULTS / "test_metrics_pre_lockdown.csv", index=False)
lock_tab.assign(period="lockdown").to_csv(RESULTS / "test_metrics_lockdown.csv", index=False)
val_tab.to_csv(RESULTS / "val_metrics.csv", index=False)
print("\nTEST 2020 (all)\n", test_tab.round(3).to_string(index=False))
print("\nTEST pre-lockdown\n", pre_tab[["model", "n", "MAE", "skill", "skill_lo", "skill_hi"]].round(3).to_string(index=False))
print("\nTEST lockdown\n", lock_tab[["model", "n", "MAE", "skill", "skill_lo", "skill_hi"]].round(3).to_string(index=False))
SUMMARY["best_hyperparameters"] = BEST_HP
SUMMARY["best_model_on_validation"] = BEST

## 4. Stability check: rolling-origin validation inside the training period
Train on all earlier years, validate on the next year, for 2017 and 2018, using the selected hyperparameters.

In [ ]:
ro = []
for label, tr_end, va_end in [("2017", "2016-12-31", "2017-12-31"), ("2018", "2017-12-31", "2018-12-31")]:
    ftr, fva, _ = split(F, tr_end, va_end, va_end)
    kn, _ = select_numeric(ftr, FEATURE_SETS[MAIN_SET])
    cols = cols_for(ftr, MAIN_SET, kn)
    hi_f = float(np.quantile(ftr["y"], HIGH_Q))
    preds = {"persistence": persistence(fva), "city-month": city_month_predict(city_month_fit(ftr), fva)}
    for n in MODEL_NAMES:
        preds[n] = fit_model(n, BEST_HP[n], cols, ftr).predict(fva[cols])
    t = evaluate(fva, preds, hi_f, with_ci=False)
    t.insert(0, "fold", f"train<= {tr_end[:4]}, validate {label}")
    ro.append(t)
ro = pd.concat(ro, ignore_index=True)
ro.to_csv(RESULTS / "rolling_origin.csv", index=False)
print(ro[["fold", "model", "n", "MAE", "RMSE", "skill"]].round(3).to_string(index=False))

## 5. Graduate depth analysis 1: lag-feature ablation
Each feature set is re-tuned on 2019 for each model family, then scored once on the 2020 test.

In [ ]:
abl = []
for sname in FEATURE_SETS:
    cols = cols_for(train, sname, KEPT_NUM)
    for n in MODEL_NAMES:
        if sname == MAIN_SET:
            hp, model, ptest = BEST_HP[n], FITTED[n], P_test[n]
            vmae = float(TUNING[n]["val_mae"].min())
        else:
            hp, tab = tune(n, cols, train, val)
            model = fit_model(n, hp, cols, train)
            ptest, vmae = model.predict(test[cols]), float(tab["val_mae"].min())
        ci = boot_vs_persistence(test, ptest, HI)
        abl.append({"feature_set": sname, "model": n, "n_features": len(cols), "val_MAE": vmae,
                    "test_MAE": mae(test["y"], ptest), "test_skill": ci["skill"][0],
                    "skill_lo": ci["skill"][1], "skill_hi": ci["skill"][2], "hyperparameters": str(hp)})
abl = pd.DataFrame(abl)
abl.to_csv(RESULTS / "ablation_lags.csv", index=False)
print(abl.drop(columns="hyperparameters").round(3).to_string(index=False))

## 6. Graduate depth analysis 2: failure-mode audit
Where does the best model fail? High-pollution days, winter episodes, particular months and cities.

In [ ]:
pb, pp = P_test[BEST], P_test["persistence"]
tm = test["target_date"].dt.month.to_numpy()
hi_t = (test["y"] >= HI).to_numpy()
groups_test = {"all test days": np.ones(len(test), bool), "high-pollution days": hi_t, "other days": ~hi_t,
               "Jan-Feb 2020": np.isin(tm, [1, 2]), "lockdown period": lock_mask}
fail = pd.concat([group_errors(test, pb, groups_test).assign(model=BEST, split="test 2020"),
                  group_errors(test, pp, groups_test).assign(model="persistence", split="test 2020")])
tv = val["target_date"].dt.month.to_numpy()
hi_v = (val["y"] >= HI).to_numpy()
groups_val = {"all validation days": np.ones(len(val), bool), "high-pollution days": hi_v,
              "other days": ~hi_v, "Nov-Dec 2019 (winter)": np.isin(tv, [11, 12])}
pv, ppv = P_val[BEST], P_val["persistence"]
fail = pd.concat([fail, group_errors(val, pv, groups_val).assign(model=BEST, split="validation 2019"),
                  group_errors(val, ppv, groups_val).assign(model="persistence", split="validation 2019")])
fail.to_csv(RESULTS / "failure_audit_groups.csv", index=False)
print(fail.round(3).to_string(index=False))

month_rows = []
for m in sorted(set(tm)):
    mask = tm == m
    month_rows.append({"month": int(m), "n": int(mask.sum()), "MAE_model": mae(test["y"][mask], pb[mask]),
                       "MAE_persistence": mae(test["y"][mask], pp[mask])})
month_tab = pd.DataFrame(month_rows)
month_tab.to_csv(RESULTS / "failure_audit_by_month.csv", index=False)

city_test = per_city_table(test, pb, pp)
city_val = per_city_table(val, pv, ppv)
city_test.to_csv(RESULTS / "per_city_test.csv", index=False)
city_val.to_csv(RESULTS / "per_city_val.csv", index=False)
print("\nper-city test skill vs persistence:\n", city_test.round(3).to_string(index=False))
SUMMARY["per_city"] = {"test_cities_better_than_persistence": int((city_test["skill"] > 0).sum()),
                       "test_cities": int(len(city_test)),
                       "val_cities_better_than_persistence": int((city_val["skill"] > 0).sum()),
                       "val_cities": int(len(city_val))}

worst = test[["City", "target_date", "y"]].copy()
worst["pred"], worst["persistence"] = pb, pp
worst["abs_err"] = np.abs(worst["pred"] - worst["y"])
worst.sort_values("abs_err", ascending=False).head(15).to_csv(RESULTS / "failure_audit_worst_days.csv", index=False)
print("\nlargest test errors:\n", worst.sort_values("abs_err", ascending=False).head(8).to_string(index=False, float_format=lambda v: f"{v:.1f}"))

## 7. Graduate depth analysis 3: prediction intervals and their coverage
90% intervals from (a) split-conformal around the best point model and (b) quantile gradient boosting
(5% and 95% quantiles). Windows: train < calibration year < test year, all chronological.

In [ ]:
cov_rows, CONF_Q = [], {}
for label, tr_end, cal_end, te_end in WINDOWS:
    wtr, wcal, wte = split(F, tr_end, cal_end, te_end)
    kn, _ = select_numeric(wtr, FEATURE_SETS[MAIN_SET])
    cols = cols_for(wtr, MAIN_SET, kn)
    hi_w = float(np.quantile(wtr["y"], HIGH_Q))
    point = fit_model(BEST, BEST_HP[BEST], cols, wtr)
    q = conformal_q(np.abs(wcal["y"].to_numpy() - point.predict(wcal[cols])))
    CONF_Q[label] = q
    yhat = point.predict(wte[cols])
    qlo = fit_quantile_gb(BEST_HP["gb"], cols, wtr, ALPHA / 2).predict(wte[cols])
    qhi = fit_quantile_gb(BEST_HP["gb"], cols, wtr, 1 - ALPHA / 2).predict(wte[cols])
    extra = {}
    if label == "2020 test":
        pre_w = (wte["target_date"] < LOCK).to_numpy()
        extra = {"pre-lockdown": pre_w, "lockdown": ~pre_w}
    cov_rows += coverage_rows(label, "split-conformal", wte, yhat - q, yhat + q, hi_w, extra)
    cov_rows += coverage_rows(label, "quantile GB", wte, np.minimum(qlo, qhi), np.maximum(qlo, qhi), hi_w, extra)
cov = pd.DataFrame(cov_rows)
cov.to_csv(RESULTS / "interval_coverage.csv", index=False)
print("conformal half-widths q (ug/m3):", {k: round(v, 1) for k, v in CONF_Q.items()})
print(cov.round(3).to_string(index=False))
SUMMARY["conformal_halfwidth"] = CONF_Q
SUMMARY["nominal_coverage"] = 1 - ALPHA

## 8. Figures

In [ ]:
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "legend.fontsize": 7})
UNIT = r"$\mu$g/m$^3$"

# Figure 1: (a) observed vs predicted on the test set, (b) MAE by month vs persistence.
y, p = test["y"].to_numpy(), pb
fig, ax = plt.subplots(1, 2, figsize=(6.6, 2.45), gridspec_kw={"width_ratios": [1, 1.1]})
ax[0].scatter(y[~hi_t], p[~hi_t], s=3, alpha=0.35, c="tab:blue", label="other days", rasterized=True)
ax[0].scatter(y[hi_t], p[hi_t], s=6, alpha=0.7, c="tab:red", label="high-pollution days", rasterized=True)
lim = float(max(y.max(), p.max()) * 1.02)
ax[0].plot([0, lim], [0, lim], "k--", lw=0.8)
ax[0].set_xlabel(f"observed next-day PM2.5 ({UNIT})")
ax[0].set_ylabel(f"predicted ({UNIT})")
ax[0].set_title(f"(a) {MODEL_LABELS[BEST]}, 2020 test")
ax[0].legend(loc="upper left", frameon=False, markerscale=2)
mt = month_tab[month_tab["month"] <= 6]
xs = np.arange(len(mt))
ax[1].bar(xs - 0.2, mt["MAE_persistence"], 0.4, label="persistence", color="tab:gray")
ax[1].bar(xs + 0.2, mt["MAE_model"], 0.4, label=MODEL_LABELS[BEST], color="tab:blue")
ax[1].set_xticks(xs)
ax[1].set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun"][: len(mt)])
ax[1].axvline(2.5, color="k", ls=":", lw=0.8)
ax[1].set_ylabel(f"MAE ({UNIT})")
ax[1].set_title("(b) MAE by target month (dotted: ~lockdown start)")
ax[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGS / "fig_errors.pdf")
fig.savefig(FIGS / "fig_errors.png", dpi=200)

# Figure 2: empirical coverage of the nominal 90% intervals.
def _get(window, group, method):
    r = cov[(cov["window"] == window) & (cov["group"] == group) & (cov["method"] == method)]
    return float(r["coverage"].iloc[0]) if len(r) else np.nan


cats_a = [("2018 backtest", "all", "2018"), ("2019 backtest", "all", "2019"),
          ("2020 test", "pre-lockdown", "2020\npre-lockdown"), ("2020 test", "lockdown", "2020\nlockdown")]
cats_b = [("2018 backtest", "high-pollution days", "2018"), ("2019 backtest", "high-pollution days", "2019"),
          ("2020 test", "high-pollution days", "2020")]
fig, ax = plt.subplots(1, 2, figsize=(6.6, 2.3), gridspec_kw={"width_ratios": [1.3, 1]})
for a, cats, title in ((ax[0], cats_a, "(a) all days"), (ax[1], cats_b, "(b) high-pollution days")):
    xs = np.arange(len(cats))
    a.bar(xs - 0.2, [_get(w, g, "split-conformal") for w, g, _ in cats], 0.4, label="split-conformal", color="tab:blue")
    a.bar(xs + 0.2, [_get(w, g, "quantile GB") for w, g, _ in cats], 0.4, label="quantile GB", color="tab:orange")
    a.axhline(1 - ALPHA, color="k", ls="--", lw=0.8)
    a.set_xticks(xs)
    a.set_xticklabels([c[2] for c in cats])
    a.set_ylim(0, 1.25)
    a.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    a.set_title(title)
ax[0].set_ylabel("empirical coverage")
ax[0].legend(frameon=False, loc="upper center", ncol=2)
fig.tight_layout()
fig.savefig(FIGS / "fig_coverage.pdf")
fig.savefig(FIGS / "fig_coverage.png", dpi=200)
if "ipykernel" in sys.modules:
    plt.show()
plt.close("all")

## 9. LaTeX tables for the paper (generated from the saved results)

In [ ]:
def _row(tab, name):
    return tab[tab["model"] == name].iloc[0]


best_mae = min(float(_row(test_tab, n)["MAE"]) for n in MODEL_NAMES)
lines = [r"\begin{tabular}{lrrrrrr}", r"\toprule",
         r"Model & MAE & RMSE & MAE$_{\mathrm{high}}$ & Skill [95\% CI] & MAE$_{\mathrm{pre}}$ & MAE$_{\mathrm{lock}}$ \\",
         r"\midrule"]
for n in ["persistence", "city-month"] + MODEL_NAMES:
    r, rp, rl = _row(test_tab, n), _row(pre_tab, n), _row(lock_tab, n)
    mae_s = fmt(r["MAE"])
    if n in MODEL_NAMES and abs(float(r["MAE"]) - best_mae) < 1e-9:
        mae_s = r"\textbf{" + mae_s + "}"
    skill_s = "0 (ref.)" if n == "persistence" else f"{r['skill']:+.3f} [{r['skill_lo']:+.3f}, {r['skill_hi']:+.3f}]"
    lines.append(f"{MODEL_LABELS[n]} & {mae_s} & {fmt(r['RMSE'])} & {fmt(r['MAE_high'])} & {skill_s} & "
                 f"{fmt(rp['MAE'])} & {fmt(rl['MAE'])} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
(TABLES / "table_main.tex").write_text("\n".join(lines) + "\n")

lines = [r"\begin{tabular}{lccc}", r"\toprule",
         r"Features & Ridge & Random forest & Grad.\ boosting \\", r"\midrule"]
for sname in FEATURE_SETS:
    cells = []
    for n in MODEL_NAMES:
        r = abl[(abl["feature_set"] == sname) & (abl["model"] == n)].iloc[0]
        cells.append(f"{r['test_MAE']:.1f} ({r['test_skill']:+.3f})")
    lines.append(f"{sname} & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]
(TABLES / "table_ablation.tex").write_text("\n".join(lines) + "\n")

def _cell(window, group, method, col):
    r = cov[(cov["window"] == window) & (cov["group"] == group) & (cov["method"] == method)]
    return float(r[col].iloc[0]) if len(r) else np.nan


rows_spec = [("2018 backtest", "2018 backtest", "all"), ("2019 backtest", "2019 backtest", "all"),
             ("2020 test", "2020 test (all)", "all"), ("2020 test", "\\ \\ pre-lockdown", "pre-lockdown"),
             ("2020 test", "\\ \\ lockdown", "lockdown")]
lines = [r"\begin{tabular}{lrcccccc}", r"\toprule",
         r" & & \multicolumn{2}{c}{Split-conformal} & \multicolumn{2}{c}{Quantile GB} & \multicolumn{2}{c}{Coverage, high days} \\",
         r"\cmidrule(lr){3-4}\cmidrule(lr){5-6}\cmidrule(lr){7-8}",
         r"Period & $n$ & Cov. & Width & Cov. & Width & Conf. & QGB \\", r"\midrule"]
for window, label, group in rows_spec:
    n_ = int(_cell(window, group, "split-conformal", "n"))
    hi_cells = ("--", "--") if group != "all" and window == "2020 test" else (
        fmt(_cell(window, "high-pollution days", "split-conformal", "coverage"), 2),
        fmt(_cell(window, "high-pollution days", "quantile GB", "coverage"), 2))
    if label == "2020 test (all)":
        hi_cells = (fmt(_cell(window, "high-pollution days", "split-conformal", "coverage"), 2),
                    fmt(_cell(window, "high-pollution days", "quantile GB", "coverage"), 2))
    lines.append(f"{label} & {n_} & {fmt(_cell(window, group, 'split-conformal', 'coverage'), 2)} & "
                 f"{fmt(_cell(window, group, 'split-conformal', 'mean_width'), 0)} & "
                 f"{fmt(_cell(window, group, 'quantile GB', 'coverage'), 2)} & "
                 f"{fmt(_cell(window, group, 'quantile GB', 'mean_width'), 0)} & {hi_cells[0]} & {hi_cells[1]} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
(TABLES / "table_intervals.tex").write_text("\n".join(lines) + "\n")

SUMMARY["runtime_seconds"] = round(time.time() - T0, 1)
SUMMARY["test_metrics_all"] = test_tab.round(4).to_dict(orient="records")
save_json(SUMMARY, RESULTS / "run_summary.json")
print(f"\nDone in {SUMMARY['runtime_seconds']} s. Results in {RESULTS}, paper tables in {TABLES}.")

## 10. Play around

In [ ]:
# Play around: compare observed PM2.5, persistence and the selected model for one city on the 2020 test set.
CITY = "Delhi"   # try any of: " ".join(KEEP)
m = (test["City"] == CITY).to_numpy()
d = test.loc[m, "target_date"]
plt.figure(figsize=(9, 3))
plt.plot(d, test.loc[m, "y"], label="observed", lw=1.5, color="black")
plt.plot(d, P_test["persistence"][m], label="persistence", lw=1, color="tab:gray")
plt.plot(d, P_test[BEST][m], label=MODEL_LABELS[BEST], lw=1, color="tab:blue")
plt.axvline(LOCK, color="red", ls=":", lw=1)
plt.ylabel("PM2.5 (ug/m3)"); plt.title(f"{CITY}: next-day PM2.5, 2020 test (red dotted: lockdown start)")
plt.legend(); plt.tight_layout(); plt.show()